# Berkshire prosecutor data: charges, outcomes, and bail
## A public-policy exploration of the three supplied extracts, 2017–2024

**Read this notebook as an examination of recorded decisions and data quality.** Charges are allegations. Repeated charges are not proof of repeated offending. Observed demographic or judge differences do not establish discrimination or the effect of an individual decision-maker.

The notebook reads the original workbooks without changing them. Both `List_Frame_1` and `List_Frame_2` are loaded; the latter is a continuation with **no header row**. Charts, aggregate tables, and a concise findings file are written to `eda_outputs/`. All charts and key tables are also embedded here.

**Reading order:** [scope](#1.-Scope,-definitions,-and-sources), [load and audit](#2.-Load-all-sheets-and-preserve-source-lineage), filings and offense mix, delays, demographics, dispositions and plea outcomes, judges, bail, linkage, and limitations. Each chart has a policy question, a stated population, and an interpretation/caution.

**Reproduce:** use Python 3.10+ with `pandas`, `numpy`, `openpyxl`, `matplotlib`, `ipython`, and `ipykernel`. Open this notebook in Jupyter or VS Code, select that environment, then **Restart Kernel and Run All**. No network connection or Drive access is needed to rerun the analysis. Run from this folder or its parent. Execution may take a few minutes while Excel is read.

## 1. Scope, definitions, and sources

| Population | Inclusion rule | What it answers |
|---|---|---|
| Filed charges | Initiated date in 2017–2024 | What entered the supplied filing system during the study period? |
| Disposed charges | Disposition date in 2017–2024 | What outcomes were recorded during the study period, including older filings? |
| Bail events | Status date in 2017–2024 | What bail/status observations were recorded during the study period? |
| Linked filing cohort | Filed in 2017–2024, linked to compatible charge dispositions by 2024-12-31 | Which supplied charges have an observed outcome by the study cutoff? |

These populations intentionally differ. The linked analysis is not a substitute for the disposition-year analysis. Dispositions in 2025 are used only in an explicitly labeled follow-up sensitivity check. A missing match is **not automatically a pending case**. Available records do not establish completeness of referrals, historical statuses, or all subsequent proceedings.

**Charge-level and case-level results are labeled separately.** A case-defendant key is `(internal case ID, defendant ID)`; charges use the source count number as well. Case-level criminal scope means at least one recorded `F`, `M+`, or `M-` charge. The representative offense uses the highest source class (`F > M+ > M- > CM`), then deterministic code/count tie-breaks. It is a broad class proxy, not a fully adjudicated most-serious-offense ranking.

**Offense class definitions.** The Trial Court complaint manual defines `F` as felony; `M+` as misdemeanor punishable by more than 100 days incarceration; `M-` as misdemeanor punishable by no incarceration or at most 100 days; `CM` as civil motor vehicle infraction. These describe authorized punishment, **not the sentence actually received**. Blank class stays Unknown. OUI remains in the criminal view when coded criminal. No class is imputed.

Primary legal references, consulted November 5, 2026:

1. [Trial Court complaint manual: explanation of F/M codes](https://www.mass.gov/doc/complaint-manual-index-a-l/download). This is a code legend, not a historical statute-by-statute validation of the source classifications.
2. [M.G.L. c.274 §1: felony and misdemeanor definitions](https://malegislature.gov/Laws/GeneralLaws/PartIV/TitleI/Chapter274/Section1).
3. [Massachusetts Criminal Procedure Rule 12](https://www.mass.gov/rules-of-criminal-procedure/criminal-procedure-rule-12-pleas-and-plea-agreements) and [Massachusetts Guide to Evidence, conviction discussion](https://www.mass.gov/doc/massachusetts-guide-to-evidence-2026-edition/download). A continued-without-finding disposition is distinct from a guilty finding; a plea label alone does not identify the final legal result. No immigration, licensing, or other collateral-consequence conclusion is inferred here.
4. [Massachusetts bail process after arraignment](https://www.mass.gov/info-details/the-bail-process-post-arraignment) and [M.G.L. c.276 §58](https://malegislature.gov/Laws/GeneralLaws/PartIV/TitleII/Chapter276/Section58). **Bail requested/set is not money paid, money collected by a prosecutor, revenue, or forfeiture.** Deposits can be returned. These workbooks contain no payment ledger.
5. [Team analysis criteria](https://docs.google.com/document/d/1Ba0FULReclAJJYfoqGe-NrArqX9EutvvRWoEKeXDgfs/edit): retain unknowns; disclose missingness; do not publish dependent metrics with more than 20% missingness in their eligible population. Demographic groups below 30 observations are omitted from comparative stacked charts; controlled comparison cells below 30 are suppressed. Volume tables retain aggregate counts.

**Taxonomy status:** the outcome crosswalk below is an explicit exploratory proposal. Original disposition fields are retained. CWOF, continued without change of plea, dismissal, and nolle prosequi remain distinguishable. Prosecutor dismissals and negotiated pleas cannot be attributed to the judge alone.

In [ ]:
from pathlib import Path
import calendar
import hashlib
import json
import platform
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from IPython.display import display, Markdown
import openpyxl

warnings.filterwarnings('ignore', category=FutureWarning)
START = pd.Timestamp('2017-01-01')
END = pd.Timestamp('2024-12-31')
FOLLOWUP_END = pd.Timestamp('2025-12-31')
MIN_CELL = 30
TOP_N = 12
DATA_DIR = Path.cwd()
if not (DATA_DIR / 'charges filed info.xlsx').exists():
    DATA_DIR = DATA_DIR / 'berkshire'
assert (DATA_DIR / 'charges filed info.xlsx').exists(), 'Run from the Berkshire folder or its parent.'
OUT = DATA_DIR / 'eda_outputs'
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
(OUT / 'tables').mkdir(exist_ok=True)
TABLES, FINDINGS, FIGURES = {}, [], []

COLORS = ['#205072','#2c8c86','#e0a13a','#b75453','#7f7099','#69a5cc','#8f9779','#c388a3','#777777','#b7c2cb']
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':150, 'font.family':'DejaVu Sans',
                     'font.size':10, 'axes.titlesize':14, 'axes.labelsize':10,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.prop_cycle':plt.cycler(color=COLORS)})
pd.set_option('display.max_columns', 16)
pd.set_option('display.max_colwidth', 65)

def note(text):
    FINDINGS.append(text)
    display(Markdown('**Readout.** ' + text))

def table(name, frame, rows=20, digits=2):
    frame = frame.to_frame() if isinstance(frame, pd.Series) else frame.copy()
    TABLES[name] = frame
    frame.to_csv(OUT / 'tables' / f'{name}.csv', index=True)
    display(frame.head(rows).round(digits))
    if len(frame) > rows:
        display(Markdown(f'Showing {rows:,} of {len(frame):,} rows. Complete table: `eda_outputs/tables/{name}.csv`.'))
    return frame

def finish(fig, name, caption):
    fig.text(.01, .01, caption, fontsize=8, color='#4f5c66', wrap=True)
    fig.tight_layout(rect=(0,.07,1,.98))
    fig.savefig(OUT / 'figures' / f'{name}.png', bbox_inches='tight')
    FIGURES.append(name)
    plt.show()
    plt.close(fig)

def barh(series, title, xlabel='Recorded charge rows', color=None):
    series = series.sort_values()
    fig, ax = plt.subplots(figsize=(11,max(4, .36*len(series)+1.3)))
    ax.barh([str(x) for x in series.index], series.values, color=color or COLORS[0])
    ax.set(title=title, xlabel=xlabel)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))
    for i, value in enumerate(series.values):
        ax.text(value, i, f' {value:,.0f}', va='center', fontsize=9)
    ax.margins(x=.18)
    return fig, ax

def stacked(frame, title, percent=False):
    frame = frame.copy()
    if percent:
        frame = frame.div(frame.sum(axis=1).replace(0,np.nan), axis=0)*100
    fig, ax = plt.subplots(figsize=(12,6))
    frame.plot.bar(stacked=True, ax=ax, color=COLORS[:len(frame.columns)], width=.8)
    ax.set(title=title, ylabel='Share within group (%)' if percent else 'Recorded charge rows', xlabel='')
    ax.tick_params(axis='x', labelrotation=35)
    if percent: ax.set_ylim(0,100)
    ax.legend(title='', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
    return fig, ax

def heatmap(frame, title, percent=False, annotate=False, cmap='Blues'):
    frame=frame.dropna(how='all').dropna(axis=1,how='all')
    if frame.empty:
        fig,ax=plt.subplots(figsize=(10,4))
        ax.text(.5,.5,'Insufficient data after the reporting threshold',ha='center',va='center',transform=ax.transAxes)
        ax.set_title(title);ax.set_axis_off()
        return fig,ax
    fig, ax = plt.subplots(figsize=(max(9,.72*len(frame.columns)),max(4,.4*len(frame)+1.8)))
    arr = frame.to_numpy(dtype=float)
    im = ax.imshow(np.ma.masked_invalid(arr), aspect='auto', cmap=cmap,
                   vmin=0)
    ax.set_xticks(range(len(frame.columns)), [str(c) for c in frame.columns], rotation=40, ha='right')
    ax.set_yticks(range(len(frame)), [str(i) for i in frame.index])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label='Percent' if percent else 'Charge rows')
    if annotate:
        midpoint = np.nanmax(arr)/2
        for i in range(len(frame)):
            for j in range(len(frame.columns)):
                value=arr[i,j]
                ax.text(j,i, '—' if np.isnan(value) else f'{value:.0f}', ha='center',va='center',
                        color='white' if value>midpoint else '#263746',fontsize=8)
    return fig, ax

def in_window(series, end=END):
    return series.between(START,end)

def parse_date(series):
    # Keep raw columns separately. Malformed centuries are flagged, not guessed.
    parsed = pd.to_datetime(series, format='mixed', errors='coerce')
    years = parsed.dt.year
    return parsed.where(years.between(1900,2026)).astype('datetime64[ns]')

def normalize_id(series):
    return series.map(lambda x: str(int(x)) if isinstance(x,(float,np.floating)) and pd.notna(x) and x.is_integer()
                      else str(x).strip() if pd.notna(x) else pd.NA).astype('string')

def load_workbook(path):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    frames, header = [], None
    for sheet in wb:
        values = sheet.values
        first = next(values)
        if header is None:
            header = [str(v).strip() for v in first]
            rows, first_excel_row = list(values), 2
        elif [str(v).strip() for v in first] == header:
            rows, first_excel_row = list(values), 2
        else:
            rows, first_excel_row = [first] + list(values), 1
        part = pd.DataFrame(rows, columns=header)
        part['source_sheet'] = sheet.title
        part['source_row'] = np.arange(first_excel_row, first_excel_row+len(part))
        frames.append(part)
    wb.close()
    frame = pd.concat(frames, ignore_index=True)
    for column in header:
        frame[column] = frame[column].map(lambda x: x.strip() if isinstance(x,str) else x)
        frame[column] = frame[column].replace(r'^\s*$', pd.NA, regex=True)
    return frame

## 2. Load all sheets and preserve source lineage

**Question:** did the import include every sheet and preserve the original records? Exact duplicate rows are flagged, but retained in the primary analysis because repeated records can represent separately recorded proceedings. Sensitivity counts are reported below. Source hashes are checked again at the end.

In [ ]:
paths = {'filed':DATA_DIR/'charges filed info.xlsx',
         'disposed':DATA_DIR/'disposed info.xlsx', 'bail':DATA_DIR/'bail info.xlsx'}
source_hashes = {name:hashlib.sha256(path.read_bytes()).hexdigest() for name,path in paths.items()}
raw = {name:load_workbook(path) for name,path in paths.items()}
inventory=[]
for name, frame in raw.items():
    cols=[c for c in frame if not c.startswith('source_')]
    frame['exact_duplicate'] = frame.duplicated(cols, keep='first')
    for sheet, sub in frame.groupby('source_sheet', sort=False):
        inventory.append({'extract':name,'sheet':sheet,'rows':len(sub),
                          'source_columns':len(cols),'repeated_export_rows':int(sub.exact_duplicate.sum())})
table('source_inventory',pd.DataFrame(inventory))
assert len(raw['filed']) == 77250 and len(raw['disposed']) == 72896 and len(raw['bail']) == 25420
assert raw['filed'].query("source_sheet == 'List_Frame_2'").source_row.min() == 1
note('Both continuation sheets are included without losing their first data row. Primary results retain flagged repeated rows; none of the original Excel files is written to.')

## 3. Standardize names, dates, demographics, and scope

**Purpose:** use readable names while retaining source values. `U` is explicit Unknown; blank is Missing. Only unambiguous common race codes are decoded. Rare codes remain labeled with their raw code pending a Berkshire codebook. Recorded sex is not a complete measure of gender identity; `N` is left unverified.

In [ ]:
filed = raw['filed'].rename(columns={'CaseNo':'case_id','Id Prsn Dfndnt':'person_id',
 'Offense Dt':'offense_raw','Initiated Dt':'initiated_raw','Chrg Cnt':'count_no',
 'Chrg Code':'charge_code','Chrg Desc':'charge_description','Offns Class':'offense_class',
 'Race':'race_code','Gender':'sex_code','Ofcr Agncy':'agency','Current or Last Court':'court'}).copy()
disposed = raw['disposed'].rename(columns={'Nmbr Cs':'case_id','Id Prsn Dfndnt':'person_id',
 'Offense Dt':'offense_raw','Initiated Dt':'initiated_raw','Chrg Cnt':'count_no',
 'Chrg Code':'charge_code','Chrg Desc':'charge_description','Offns Class':'offense_class',
 'Race':'race_code','Gender':'sex_code','Officer Agncy':'agency','Court':'court',
 'Judge':'judge','Disp Dt':'disposed_raw','Disp Type':'disp_type','Disp Desc':'disp_description',
 'Sentence Translation':'sentence_translation'}).copy()
bail = raw['bail'].rename(columns={'Nmbr Cs':'case_id','Race':'race_code','Sex':'sex_code',
 'Date Sts':'bail_raw','Desc Dfndnt Sts':'bail_status','Bail Req':'requested','Bail Set':'set_amount',
 'Case Type':'case_type','Crime Type':'crime_type'}).copy()
race_map={'W':'White','B':'Black','H':'Hispanic (source race field)','A':'Asian','U':'Unknown (U)'}
sex_map={'M':'Male','F':'Female','U':'Unknown (U)'}
date_audit=[]
for name, frame in [('filed',filed),('disposed',disposed),('bail',bail)]:
    frame['case_id']=normalize_id(frame.case_id)
    if 'person_id' in frame:
        frame['person_id']=normalize_id(frame.person_id)
        frame['case_person']=frame.case_id+' | '+frame.person_id
        frame['charge_code']=normalize_id(frame.charge_code)
        frame['count_no']=normalize_id(frame.count_no)
        frame['offense_class']=frame.offense_class.fillna('Unknown')
        frame['criminal_charge']=frame.offense_class.isin(['F','M+','M-'])
        frame['placeholder_charge']=frame.charge_code.isin(['666666','999999'])
    for source, target in [('offense_raw','offense_date'),('initiated_raw','initiated_date'),
                           ('disposed_raw','disposition_date'),('bail_raw','bail_date')]:
        if source in frame:
            frame[target]=parse_date(frame[source])
            date_audit.append({'extract':name,'field':source,'blank':int(frame[source].isna().sum()),
              'invalid_or_outside_1900_2026':int((frame[source].notna()&frame[target].isna()).sum()),
              'valid_min':frame[target].min(),'valid_max':frame[target].max()})
    frame['race']=frame.race_code.map(lambda x: 'Missing' if pd.isna(x) else race_map.get(x,f'Code {x} (unverified)'))
    frame['sex']=frame.sex_code.map(lambda x: 'Missing' if pd.isna(x) else sex_map.get(x,f'Code {x} (unverified)'))
filed['filing_year']=filed.initiated_date.dt.year
filed['filing_month']=filed.initiated_date.dt.month
disposed['disposition_year']=disposed.disposition_date.dt.year
f=filed.loc[in_window(filed.initiated_date)].copy()
d=disposed.loc[in_window(disposed.disposition_date)].copy()
b=bail.loc[in_window(bail.bail_date)].copy()
for c in ['requested','set_amount']:
    b[c]=pd.to_numeric(b[c],errors='coerce')
table('date_validation',pd.DataFrame(date_audit))
cohorts=pd.DataFrame({'rows':[len(f),len(d),len(b)],
 'case_ids':[f.case_id.nunique(),d.case_id.nunique(),b.case_id.nunique()],
 'case_defendant_pairs':[f.case_person.nunique(),d.case_person.nunique(),np.nan]},
 index=['Filed in 2017–2024','Disposed in 2017–2024','Bail dated 2017–2024'])
table('analysis_populations',cohorts)
coverage=[]
for name, frame, fields in [('Filed',f,['race_code','sex_code','offense_date','offense_class','agency','court']),
 ('Disposed',d,['race_code','sex_code','judge','disp_type','disp_description','sentence_translation','offense_date']),
 ('Bail',b,['race_code','sex_code','requested','set_amount','crime_type'])]:
    for field in fields:
        missing=frame[field].isna() | (frame[field].eq('Unknown') if field=='offense_class' else False)
        coverage.append({'extract':name,'field':field,'rows':len(frame),'missing':int(missing.sum()),
                         'missing_pct':missing.mean()*100})
coverage=pd.DataFrame(coverage)
table('coverage_by_population',coverage,rows=30)
fig,ax=barh(coverage.set_index(coverage.extract+' / '+coverage.field).missing_pct,
            'Coverage gaps differ sharply by field', 'Missing within stated population (%)')
ax.axvline(20,color=COLORS[3],ls='--',label='Team threshold: 20%');ax.legend()
finish(fig,'01_field_coverage','Missingness across all rows; conditional sentencing/cash-bail denominators are assessed separately below.')
note(f'The study contains {len(f):,} filed charge rows, {len(d):,} disposed charge rows, and {len(b):,} bail events. Dates outside 1900–2026 are flagged, never repaired by guessing a century.')

## 4. How did filing volume and offense class change?

**Why this graph:** a public-policy reader needs to separate changes in caseload from changes in charge intensity. Source initiation date defines filing year. This is recorded prosecution activity, not a population crime rate. Full-extract counts include civil motor vehicle infractions; the criminal-only comparison excludes `CM` and Unknown class.

In [ ]:
annual=f.groupby('filing_year').agg(charge_rows=('case_id','size'),cases=('case_person','nunique'),
                                people=('person_id','nunique'))
annual['charges_per_case']=annual.charge_rows/annual.cases
annual['case_yoy_pct']=annual.cases.pct_change()*100
annual['case_change_vs_2017_pct']=(annual.cases/annual.loc[2017,'cases']-1)*100
criminal_cases=f.loc[f.criminal_charge].groupby('filing_year').case_person.nunique()
annual['cases_with_criminal_charge']=criminal_cases
table('annual_filing_volume',annual)
fig,axs=plt.subplots(1,2,figsize=(13,5))
for c,label in [('charge_rows','Charge rows'),('cases','Case-defendant pairs'),('cases_with_criminal_charge','Cases with ≥1 criminal charge')]:
    axs[0].plot(annual.index,annual[c],marker='o',label=label)
axs[0].set(title='Caseload and charge volume',xlabel='Initiation year',ylabel='Count');axs[0].legend(fontsize=8)
axs[1].plot(annual.index,annual.charges_per_case,marker='o',color=COLORS[1])
axs[1].set(title='Charge intensity per case',xlabel='Initiation year',ylabel='Charge rows / case-defendant pair')
finish(fig,'02_filing_volume','Charge rows and case-defendant pairs are different units. No population denominator or causal COVID/policy attribution.')
class_order=['F','M+','M-','CM','Unknown']
classes=pd.crosstab(f.filing_year,f.offense_class).reindex(columns=class_order,fill_value=0)
table('offense_class_annual',classes)
table('offense_class_overall',f.offense_class.value_counts().rename('charges').to_frame().assign(
    percent=lambda x:x.charges/len(f)*100))
fig,ax=stacked(classes,'Offense-class composition by initiation year',percent=True)
finish(fig,'03_offense_class','F felony; M+ >100-day authorized incarceration; M− ≤100 days or none; CM civil motor vehicle. Source coding is retained.')
note(f'Case volume changed {annual.loc[2024,"case_change_vs_2017_pct"]:.1f}% from 2017 to 2024. The class chart describes charges, not the share of defendants convicted or incarcerated.')

## 5. What are the actual charge descriptions?

**Why this graph:** show which allegations drive workload before creating broad offense groups. A statutory charge code is used as the comparison key; its most common recorded description supplies a readable label. Every literal description is retained in a complete frequency CSV. Different text attached to one code is audited instead of silently treated as a different offense.

In [ ]:
literal=f.groupby('charge_description',dropna=False).agg(charge_rows=('case_id','size'),
 cases=('case_person','nunique'),people=('person_id','nunique')).sort_values('charge_rows',ascending=False)
literal['share_of_filed_charge_rows_pct']=literal.charge_rows/len(f)*100
table('all_unique_charge_descriptions',literal,rows=25)
canonical=f.groupby('charge_code').charge_description.agg(lambda s:s.dropna().mode().iloc[0] if s.notna().any() else 'Unknown description')
labels={k: f'{k} · {str(v)[:62]}' for k,v in canonical.items()}
for frame in [f,d,filed,disposed]:
    frame['offense_label']=frame.charge_code.map(labels).fillna(frame.charge_code)
code_counts=f.charge_code.value_counts()
top_codes=code_counts.head(TOP_N).index.tolist()
fig,ax=barh(code_counts.head(TOP_N).rename(index=labels),'The allegations that dominate recorded filings')
finish(fig,'04_top_charges','All filed charge rows, 2017–2024. Labels use the most frequent source description per code; repeated allegations are not convictions.')
variants=f.groupby('charge_code').agg(description_variants=('charge_description','nunique'),
 charges=('case_id','size')).sort_values(['description_variants','charges'],ascending=False)
table('charge_description_variants_by_code',variants)
table('charge_description_year_counts',pd.crosstab(f.charge_description,f.filing_year),rows=15)
note(f'There are {f.charge_description.nunique():,} literal descriptions and {f.charge_code.nunique():,} charge codes in the filing window. The top {TOP_N} codes account for {code_counts.head(TOP_N).sum()/len(f):.1%} of charge rows. Text variation and placeholder codes should not be interpreted as new criminal conduct.')

## 6. Which charges changed by year and month?

**Why these graphs:** annual counts reveal workload; within-year shares reveal composition. The 96-month chart preserves chronology, while the pooled January–December chart measures recorded initiation seasonality across eight years. It is not offense-date seasonality or a forecast.

In [ ]:
year_code=pd.crosstab(f.charge_code,f.filing_year)
year_share=year_code.div(f.groupby('filing_year').size(),axis=1)*100
table('charge_code_year_counts',year_code,rows=20)
table('charge_code_year_shares_pct',year_share,rows=20)
fig,ax=heatmap(year_share.reindex(top_codes).rename(index=labels),
              'How the leading charge codes changed as a share of filings',percent=True,annotate=True)
finish(fig,'05_charge_year_heatmap','Each cell: code rows / all filed charge rows in that year (%). Top codes selected across the full period, not independently each year.')
monthly_index=pd.period_range('2017-01','2024-12',freq='M')
monthly=pd.crosstab(f.initiated_date.dt.to_period('M'),f.charge_code).reindex(monthly_index,fill_value=0)
table('charge_code_96_month_counts',monthly,rows=12)
fig,ax=plt.subplots(figsize=(13,5))
for c in top_codes[:5]: ax.plot(monthly.index.to_timestamp(),monthly[c],label=labels[c][:53],lw=1.4)
ax.set(title='Eight years of monthly filing activity',xlabel='Initiation month',ylabel='Charge rows')
ax.legend(bbox_to_anchor=(1.01,1),loc='upper left',fontsize=8)
finish(fig,'06_charge_monthly_timeline','Monthly counts for the five leading codes; years are not pooled in this chart. Filing changes can reflect recording and charging practices.')
pooled=pd.crosstab(f.charge_code,f.filing_month).reindex(columns=range(1,13),fill_value=0)
table('charge_code_pooled_month_counts',pooled,rows=20)
pooled_shares=pooled.div(f.groupby('filing_month').size(),axis=1)*100
fig,axs=plt.subplots(1,2,figsize=(13,5))
month_counts=f.groupby('filing_month').size().reindex(range(1,13),fill_value=0)
axs[0].bar(range(1,13),month_counts,color=COLORS[0]);axs[0].set_xticks(range(1,13),list(calendar.month_abbr)[1:])
axs[0].set(title='Pooled initiation month: total volume',ylabel='Charge rows across eight years')
days=pd.Series(pd.date_range(START,END)).dt.month.value_counts().reindex(range(1,13))
axs[1].bar(range(1,13),month_counts/days,color=COLORS[1]);axs[1].set_xticks(range(1,13),list(calendar.month_abbr)[1:])
axs[1].set(title='Adjusting for unequal calendar-month lengths',ylabel='Charge rows / calendar day')
finish(fig,'07_pooled_month_volume','All eight years pooled. Calendar-day adjustment does not adjust for court holidays, working days, or office closure schedules.')
hm=pooled_shares.reindex(top_codes).rename(index=labels).rename(columns=dict(enumerate(list(calendar.month_abbr)[1:],1)))
fig,ax=heatmap(hm,'Does the charge mix vary by calendar month?',percent=True,annotate=True)
finish(fig,'08_pooled_month_mix','Each cell: charge-code share of all filings in that calendar month, pooling 2017–2024. This is not within-code monthly prevalence.')
note(f'{calendar.month_name[int(month_counts.idxmax())]} has the most pooled charge rows. Compare the daily-adjusted panel before attributing this pattern to seasonality.')

## 7. How long from alleged offense to initiation?

**Why these graphs:** averages can be dominated by old allegations, so show mean, median, quantiles, and the tail. The unit is a filed charge row; a companion case-defendant estimate uses the earliest valid offense date and earliest initiation date per case. Dates before 1900 are invalid for this analysis; legitimate older allegations remain. Missing/negative intervals are disclosed, not converted to zero.

In [ ]:
f['offense_to_initiation_days']=(f.initiated_date-f.offense_date).dt.days
delay=f.offense_to_initiation_days.where(f.offense_to_initiation_days.ge(0)).dropna()
delay_summary=delay.describe(percentiles=[.25,.5,.75,.9,.95,.99]).to_frame('charge_weighted_days')
table('offense_to_initiation_summary',delay_summary)
delay_by_year=f.loc[f.offense_to_initiation_days.ge(0)].groupby('filing_year').offense_to_initiation_days.agg(
 n='size',mean='mean',median='median',p90=lambda s:s.quantile(.9),max='max')
table('offense_to_initiation_by_year',delay_by_year)
case_delays=f.groupby('case_person').agg(first_offense=('offense_date','min'),first_initiation=('initiated_date','min'))
case_delays['days']=(case_delays.first_initiation-case_delays.first_offense).dt.days
table('offense_to_initiation_case_weighted',case_delays.days.where(case_delays.days.ge(0)).describe().to_frame('days'))
fig,axs=plt.subplots(1,2,figsize=(13,5))
axs[0].hist(np.log1p(delay),bins=55,color=COLORS[0])
axs[0].set(title='Most delays and the long tail',xlabel='log(1 + days)',ylabel='Charge rows')
for c in ['mean','median','p90']:axs[1].plot(delay_by_year.index,delay_by_year[c],marker='o',label=c)
axs[1].set(title='Mean, median, and 90th percentile',xlabel='Initiation year',ylabel='Days');axs[1].legend()
finish(fig,'09_offense_to_initiation','Valid nonnegative intervals only. This is offense-to-initiation, not arrest-to-charge or referral-to-decision time.')
delay_class=f.loc[f.offense_to_initiation_days.ge(0)].groupby('offense_class').offense_to_initiation_days.agg(
 n='size',mean='mean',median='median',p90=lambda s:s.quantile(.9))
table('offense_to_initiation_by_class',delay_class)
table('delay_sensitivity',pd.DataFrame({'mean_days':[delay.mean(),delay[delay<=3650].mean()],
 'median_days':[delay.median(),delay[delay<=3650].median()], 'n':[len(delay),int((delay<=3650).sum())]},
 index=['All valid nonnegative intervals','Sensitivity: at most 10 years']))
note(f'Mean offense-to-initiation delay is {delay.mean():,.1f} days; median {delay.median():,.0f}; 90th percentile {delay.quantile(.9):,.0f}. '
     f'{f.offense_date.isna().sum():,} rows lack a usable offense date and {(f.offense_to_initiation_days<0).sum():,} have negative intervals. Long valid intervals remain in the primary mean; the 10-year sensitivity is explicitly secondary.')

## 8. Which recorded people account for the most charges?

**Why these graphs:** distinguish concentration across defendants from many counts on a single case. Internal defendant IDs are shown only in the requested aggregate ranking table, with no names or narratives. Repeated records are retained in the primary ranking; a sensitivity removes exact repeated export rows. This is not a recidivism measure, and county IDs cannot identify the same person in another jurisdiction.

In [ ]:
person_workload=f.groupby('person_id').agg(charge_rows=('case_id','size'),cases=('case_person','nunique'),
 codes=('charge_code','nunique'),active_years=('filing_year','nunique')).sort_values('charge_rows',ascending=False)
without_repeats=f.loc[~f.exact_duplicate].groupby('person_id').size()
person_workload['rows_without_exact_repeats']=without_repeats
table('highest_recorded_charge_counts_by_defendant_id',person_workload,rows=20)
fig,axs=plt.subplots(1,2,figsize=(13,5))
top_people=person_workload.head(15)
axs[0].barh([f'Rank {i}' for i in range(15,0,-1)],top_people.charge_rows.iloc[::-1],color=COLORS[0])
axs[0].set(title='Highest recorded charge counts',xlabel='Charge rows per defendant ID')
ordered=np.sort(person_workload.charge_rows.values)
x=np.arange(len(ordered)+1)/len(ordered);y=np.r_[0,np.cumsum(ordered)/ordered.sum()]
axs[1].plot(x*100,y*100,lw=2);axs[1].plot([0,100],[0,100],ls='--',color='#999')
axs[1].set(title='Concentration of recorded charge rows',xlabel='Defendant IDs, ascending workload (%)',ylabel='Cumulative charge rows (%)')
finish(fig,'10_charge_concentration','Defendant IDs in the filing cohort. Concentration is administrative workload, not a finding of guilt or a risk score.')
top_one=max(1,int(np.ceil(len(person_workload)*.01)))
note(f'The highest-charge defendant ID has {person_workload.iloc[0].charge_rows:,.0f} recorded charge rows across {person_workload.iloc[0].cases:,.0f} cases. '
     f'The top 1% of defendant IDs account for {person_workload.head(top_one).charge_rows.sum()/len(f):.1%} of filed charge rows.')

## 9. Race and recorded sex: workload composition

**Why these graphs:** present both absolute counts and within-group composition. Larger bars can reflect population size, policing, referrals, charge intensity, and recording. No resident-population denominator is available, so these are not per-capita prosecution rates. Small comparative groups are suppressed, with Unknown and Missing kept distinct when large enough.

In [ ]:
for group in ['race','sex']:
    groups=f[group].value_counts(); eligible=groups[groups>=MIN_CELL].index
    ct=pd.crosstab(f[group],f.offense_class).reindex(columns=class_order,fill_value=0)
    table(f'filing_class_by_{group}',ct)
    fig,ax=stacked(ct.loc[eligible],f'Offense classes by {group}: volume')
    finish(fig,f'11_{group}_class_counts',f'Filed charge rows. Comparative groups require at least {MIN_CELL} rows; rare raw codes are preserved in the source tables.')
    fig,ax=stacked(ct.loc[eligible],f'Offense classes within {group}',percent=True)
    finish(fig,f'12_{group}_class_shares','Denominator: all filed charge rows within each displayed group. Descriptive composition, not population exposure or a causal disparity estimate.')
    mix=pd.crosstab(f.charge_code,f[group])
    table(f'charge_code_by_{group}',mix,rows=15)
    shares=mix.div(f[group].value_counts(),axis=1)*100
    display_mix=shares.reindex(top_codes).loc[:,eligible]
    display_mix=display_mix.where(mix.reindex(top_codes).loc[:,eligible]>=MIN_CELL).rename(index=labels)
    assert display_mix.notna().any().any(), 'The demographic heatmap should retain eligible common-charge cells.'
    fig,ax=heatmap(display_mix,f'Leading charge-code shares within {group}',percent=True,annotate=True)
    finish(fig,f'13_{group}_charge_mix','Each cell: code rows / all filed charge rows in that demographic group (%). Cells with fewer than 30 charge rows are suppressed (blank).')
    year_mix=pd.crosstab(f.filing_year,f[group])
    table(f'filing_year_by_{group}',year_mix)
    fig,ax=stacked(year_mix.loc[:,eligible],f'Recorded {group} composition of filings by year',percent=True)
    finish(fig,f'14_{group}_annual_mix','Rare groups under 30 rows overall omitted from this comparative chart; complete raw counts remain in the CSV. Demographics are charge-weighted.')
note('Both count and percentage charts are needed: a large count is not necessarily a high within-group share. No causal claim or population-adjusted racial/gender ranking is made.')

## 10. What disposition types were recorded, overall and by year?

**Why these graphs:** establish the original outcomes before interpreting them. The year is disposition year, not filing year. Original `Disp Type` and `Disp Desc` are not interchangeable; their joint table exposes disagreements. These charge-level shares include civil matters; the criminal-only companion is explicitly separate.

In [ ]:
raw_outcomes=d.disp_type.fillna('Missing').value_counts().to_frame('charge_rows')
raw_outcomes['percent']=raw_outcomes.charge_rows/len(d)*100
table('raw_disposition_type_overall',raw_outcomes,rows=40)
disp_year=pd.crosstab(d.disposition_year,d.disp_type.fillna('Missing'))
table('raw_disposition_type_by_year',disp_year,rows=10)
fig,ax=barh(raw_outcomes.charge_rows.head(15),'Recorded disposition types, 2017–2024')
finish(fig,'15_raw_disposition_types','Disposed charge rows by disposition date. A Plea label alone does not establish conviction, custody, or CWOF.')
leading_disp=raw_outcomes.head(7).index
annual_disp=disp_year.reindex(columns=leading_disp,fill_value=0)
annual_disp['Other recorded types']=disp_year.sum(axis=1)-annual_disp.sum(axis=1)
fig,ax=stacked(annual_disp,'Original disposition types by year: counts')
finish(fig,'16_raw_outcomes_annual_counts','Disposition-year counts include cases initiated before 2017. Counts can change with case mix, backlog, and data coverage.')
fig,ax=stacked(annual_disp,'Original disposition types by year: shares',percent=True)
finish(fig,'17_raw_outcomes_annual_shares','Each year sums to 100% of disposed charge rows. This is not a filing-cohort resolution or conviction rate.')
cross=pd.crosstab(d.disp_type.fillna('Missing'),d.disp_description.fillna('Missing'))
table('disposition_type_vs_description_audit',cross,rows=30)
table('criminal_raw_disposition_types',d.loc[d.criminal_charge].disp_type.value_counts().to_frame('charge_rows'),rows=30)
note(f'The most frequent raw label is {raw_outcomes.index[0]}, covering {raw_outcomes.iloc[0].percent:.1f}% of disposed charge rows. Original type/description combinations remain available for client review rather than being overwritten.')

## 11. Plea does not always mean jail or a guilty finding

**Why these graphs:** the source sentence translation reveals legally important detail hidden inside the Plea label. The exploratory classification below is fully visible. Positive recorded actual-jail terms corroborate a custody marker but do not establish time physically served. Blank translation is unresolved information, not zero custody. Contradictory CWOF/custody markers are quarantined as conflicts.

In [ ]:
def classify_outcome(row):
    typ=str(row.get('disp_type','')).strip()
    desc=str(row.get('disp_description','')).strip()
    sentence=str(row.get('sentence_translation','')).strip()
    actual_values=[pd.to_numeric(row.get(c),errors='coerce') for c in
                   ['Min Actl Jaill Impsd Term','Max Actl Jail Imp Term']]
    positive_actual=any(pd.notna(x) and x>0 for x in actual_values)
    text_custody=sentence in ['Jail','Jail (Time Served)','Split Sentence','Committed']
    cwof=sentence=='Continued w/o Finding' or typ=='Continued Without Finding'
    if cwof and (positive_actual or text_custody): return 'Conflicting CWOF / custody markers'
    if typ=='Plea':
        if cwof:return 'Plea · CWOF'
        if text_custody or positive_actual:return 'Plea · custody marker'
        if sentence in ['Probation','Suspended Sentence']:return 'Plea · probation / suspended'
        if sentence in ['Fines','Court Cost/Restitution','Community Service','House Arrest']:return 'Plea · other sentence marker'
        if sentence in ['Filed','Filed w/o Change of Plea']:return 'Plea · filed'
        if sentence=='Continued w/o Change of Plea':return 'Continued without change of plea'
        return 'Plea · sentence unresolved'
    if typ.startswith('Dismissed'):return 'Dismissed (all source variants)'
    if typ=='Nolle Prosequi':return 'Nolle prosequi'
    if typ=='Trial':
        if desc in ['Not Guilty','Required Finding of Not Guilty','Directed Verdict']:return 'Trial · not guilty marker'
        if desc=='Guilty':return 'Trial · guilty marker'
        return 'Trial · result unresolved'
    if typ=='Guilty':return 'Guilty · route unspecified'
    if typ in ['Not Guilty','Required Finding of Not Guilty','Directed Verdict']:return 'Not guilty / directed verdict'
    if cwof:return 'CWOF · explicit type'
    if typ=='Continued Without Change of Plea':return 'Continued without change of plea'
    if typ in ['Responsible','Not Responsible','Converted to Civil']:return 'Civil / responsibility result'
    return 'Other / review required'

disposed['outcome_detail']=disposed.apply(classify_outcome,axis=1)
d=disposed.loc[in_window(disposed.disposition_date)].copy()
d['custody_marker']=d.sentence_translation.isin(['Jail','Jail (Time Served)','Split Sentence','Committed']) | (
    pd.to_numeric(d['Max Actl Jail Imp Term'],errors='coerce').gt(0)) | (
    pd.to_numeric(d['Min Actl Jaill Impsd Term'],errors='coerce').gt(0))
d['cwof_marker']=d.outcome_detail.isin(['Plea · CWOF','CWOF · explicit type'])
pleas=d.loc[d.disp_type.eq('Plea')].copy()
plea_table=pleas.groupby('outcome_detail').agg(charge_rows=('case_id','size'),cases_with_marker=('case_person','nunique'))
plea_table['percent_of_plea_rows']=plea_table.charge_rows/len(pleas)*100
table('plea_outcome_decomposition',plea_table.sort_values('charge_rows',ascending=False),rows=30)
table('plea_sentence_translation_raw',pleas.sentence_translation.fillna('Missing').value_counts().to_frame('charge_rows'),rows=30)
raw_plea_cwof=pleas.sentence_translation.eq('Continued w/o Finding').fillna(False)
text_custody=pleas.sentence_translation.isin(['Jail','Jail (Time Served)','Split Sentence','Committed'])
plea_evidence=pd.Series({'all_plea_rows':len(pleas),'CWOF_translation_rows':int(raw_plea_cwof.sum()),
 'CWOF_translation_with_positive_actual_jail_field':int(pleas.outcome_detail.eq('Conflicting CWOF / custody markers').sum()),
 'explicit_custodial_translation_rows':int(text_custody.sum()),
 'CWOF_without_conflicting_custody_field':int(pleas.outcome_detail.eq('Plea · CWOF').sum()),
 'plea_sentence_unresolved':int(pleas.outcome_detail.eq('Plea · sentence unresolved').sum())},name='charge_rows')
table('plea_marker_evidence_audit',plea_evidence)
fig,ax=barh(plea_table.charge_rows,'What is inside the Plea label?')
finish(fig,'18_plea_decomposition','Disposed 2017–2024 Plea charge rows. Custody is a recorded sentence marker; filed/unresolved labels are not inferred to mean noncustodial.')
plea_year=pd.crosstab(pleas.disposition_year,pleas.outcome_detail)
table('plea_outcome_decomposition_by_year',plea_year)
fig,ax=stacked(plea_year,'How recorded plea results changed by year',percent=True)
finish(fig,'19_plea_outcomes_annual','Denominator: all Plea charge rows disposed in that year, including unresolved sentence information.')
case_markers=pleas.groupby('case_person').agg(plea_charge_rows=('case_id','size'),
 any_custody_marker=('custody_marker','max'),any_cwof_marker=('cwof_marker','max'))
table('plea_case_marker_summary',pd.Series({'cases_with_plea':len(case_markers),
 'cases_with_any_plea_custody_marker':int(case_markers.any_custody_marker.sum()),
 'cases_with_any_plea_CWOF_marker':int(case_markers.any_cwof_marker.sum()),
 'cases_with_both_on_different_or_conflicting_charges':int((case_markers.any_cwof_marker&case_markers.any_custody_marker).sum())},name='case_defendant_pairs'))
note(f'Among {len(pleas):,} Plea charge rows, {raw_plea_cwof.sum():,} carry a CWOF sentence translation. '
     f'Of those, {pleas.outcome_detail.eq("Conflicting CWOF / custody markers").sum():,} also have a positive actual-jail field and are quarantined as conflicts; '
     f'{pleas.outcome_detail.eq("Plea · CWOF").sum():,} retain an unconflicted CWOF marker. '
     f'{text_custody.sum():,} have an explicit custodial sentence translation; {pleas.outcome_detail.eq("Plea · custody marker").sum():,} meet the broader custody-marker rule. '
     'Neither rule proves physical time served. Case-level markers can overlap across different charges.')

## 12. Charge description versus disposition

**Why this graph:** outcome patterns may reflect the allegations being resolved. Use the leading criminal charge codes, not a mixture of civil and criminal records. Each row sums to 100% among disposed charges of that code. Minimum code denominator is 100; small individual cells are suppressed. These are unadjusted associations, not probabilities that a newly filed charge will receive that result.

In [ ]:
def outcome_family(detail):
    if 'CWOF' in detail and not detail.startswith('Conflicting'):return 'CWOF'
    if detail=='Plea · custody marker':return 'Plea custody marker'
    if detail=='Plea · sentence unresolved':return 'Plea unresolved'
    if detail.startswith('Plea'):return 'Other plea result'
    if detail.startswith('Dismissed'):return 'Dismissed'
    if detail=='Nolle prosequi':return 'Nolle prosequi'
    if 'not guilty' in detail.lower():return 'Not guilty marker'
    return 'Other / review'
d['outcome_family']=d.outcome_detail.map(outcome_family)
dc=d.loc[d.criminal_charge].copy()
offense_outcomes=pd.crosstab(dc.charge_code,dc.outcome_family)
table('criminal_charge_code_vs_outcome_counts',offense_outcomes,rows=20)
table('literal_description_vs_raw_disposition',pd.crosstab(d.charge_description,d.disp_type),rows=15)
eligible_codes=dc.charge_code.value_counts().loc[lambda s:s>=100].head(15).index
rates=offense_outcomes.div(offense_outcomes.sum(axis=1),axis=0)*100
plot_rates=rates.loc[eligible_codes].where(offense_outcomes.loc[eligible_codes]>=MIN_CELL).rename(index=labels)
fig,ax=heatmap(plot_rates,'Criminal charge codes and recorded outcome mix',percent=True,annotate=True)
finish(fig,'20_charge_outcome_heatmap','Disposed criminal charges. Each denominator is one code; cells under 30 charges are blank. Complete count tables preserve all codes and raw descriptions.')
note('A code with more dismissals may have different evidence, screening, or case mix. No underlying facts, prior-record measure, or original referral charge are available to explain the differences.')

## 13. Race and recorded sex versus disposition

**Why these graphs:** compare the mix of recorded outcomes within demographic groups, with volume and denominators visible. Main comparative views use criminal charges and retain unresolved pleas. These are charge-weighted results: a person with many charges contributes more observations. The same-charge panels reduce one source of case-mix difference but do not adjust for prior record, facts, counsel, or all procedural differences.

In [ ]:
for group in ['race','sex']:
    counts=pd.crosstab(dc[group],dc.outcome_family)
    table(f'criminal_outcomes_by_{group}_counts',counts)
    eligible=counts.sum(axis=1).loc[lambda s:s>=MIN_CELL].index
    fig,ax=stacked(counts.loc[eligible],f'Criminal dispositions by {group}: volume')
    finish(fig,f'21_{group}_outcome_counts','Disposed criminal charge rows, 2017–2024. Totals are charge-weighted, not distinct people.')
    fig,ax=stacked(counts.loc[eligible],f'Criminal disposition mix within {group}',percent=True)
    finish(fig,f'22_{group}_outcome_shares','Within-group charge-row percentages, including unresolved outcomes. Groups below 30 rows are not compared; this is not a causal estimate.')
    source_missing=dc['race_code' if group=='race' else 'sex_code'].isna().mean()
    note(f'{group.title()} is blank on {source_missing:.1%} of the disposed criminal-charge population. Unknown/raw unverified codes remain explicit; complete-case-only comparisons are not substituted.')
    # Compare the same three frequent criminal codes without pooling them into one rate.
    same_codes=dc.charge_code.value_counts().head(3).index
    rows=[]
    for charge in same_codes:
        for label, sub in dc.loc[dc.charge_code.eq(charge)].groupby(group):
            rows.append({'charge':labels.get(charge,charge),'group':label,'n':len(sub),
                         'CWOF_marker_pct':sub.cwof_marker.mean()*100 if len(sub)>=MIN_CELL else np.nan,
                         'custody_marker_pct':sub.custody_marker.mean()*100 if len(sub)>=MIN_CELL else np.nan})
    compare=pd.DataFrame(rows)
    table(f'same_charge_{group}_comparison',compare,rows=30)
    hm=compare.pivot(index='charge',columns='group',values='CWOF_marker_pct')
    fig,ax=heatmap(hm,f'Same charge code: CWOF markers by {group}',percent=True,annotate=True)
    finish(fig,f'23_same_charge_{group}','CWOF marker / disposed criminal charge rows for that code and group. n<30 suppressed; comparison does not establish equal case facts or causal bias.')

## 14. Judges: recorded outcome patterns with case mix visible

**Why these graphs:** identify patterns worth asking about, without presenting a judge league table. The judge is the name recorded on the disposed charge; it may not identify the official responsible for every prior action. Dismissals, negotiated pleas, transfers, and prosecution choices cannot be attributed to this name alone. Use judges with at least 100 criminal charge rows; display caseload, outcome mix, and class mix together. No performance or fairness score is calculated.

In [ ]:
nonjudge=dc.judge.isna()|dc.judge.fillna('').str.contains(r'no judge|clerk|unknown|missing',case=False,regex=True)
table('nonjudge_and_missing_name_audit',dc.loc[nonjudge].judge.fillna('Missing judge').value_counts().to_frame('criminal_charge_rows'),rows=30)
judge_data=dc.loc[~nonjudge].copy()
judge_counts=judge_data.judge.value_counts()
judges=judge_counts.loc[lambda s:s>=100].head(15).index
judge_outcomes=pd.crosstab(judge_data.judge,judge_data.outcome_family).reindex(judges,fill_value=0)
table('judge_recorded_criminal_outcomes',judge_outcomes,rows=30)
fig,ax=stacked(judge_outcomes,'Outcome mix associated with the recorded judge name',percent=True)
finish(fig,'24_judge_outcome_mix','Criminal charge rows; judges need ≥100 rows. Associations reflect assignment, court, source coding, and case mix—not individual causal effects.')
judge_class=pd.crosstab(judge_data.judge,judge_data.offense_class).reindex(judges,fill_value=0)
fig,ax=stacked(judge_class,'Do those judges see the same offense-class mix?',percent=True)
finish(fig,'25_judge_class_mix','Same displayed judges and disposed criminal-charge population. Source class is only a broad case-mix measure.')
table('judge_case_load',dc.groupby('judge',dropna=False).agg(charge_rows=('case_id','size'),
 case_defendant_pairs=('case_person','nunique'),charge_codes=('charge_code','nunique')).sort_values('charge_rows',ascending=False),rows=25)
# Restrict to one common court and source class, then compare a common charge code.
cell_sizes=judge_data.groupby(['court','charge_code']).size().sort_values(ascending=False)
common_court,common_code=cell_sizes.index[0]
matched=judge_data.loc[judge_data.court.eq(common_court)&judge_data.charge_code.eq(common_code)]
judge_same=matched.groupby('judge').agg(n=('case_id','size'),CWOF=('cwof_marker','sum'),custody=('custody_marker','sum'))
judge_same=judge_same.loc[judge_same.n>=MIN_CELL].sort_values('n',ascending=False)
judge_same['CWOF_pct']=judge_same.CWOF/judge_same.n*100
judge_same['custody_marker_pct']=judge_same.custody/judge_same.n*100
table('judge_same_court_same_charge',judge_same,rows=30)
fig,ax=plt.subplots(figsize=(11,max(4,.4*len(judge_same)+1)))
ax.scatter(judge_same.CWOF_pct,range(len(judge_same)),s=55,color=COLORS[1],label='CWOF marker')
ax.scatter(judge_same.custody_marker_pct,range(len(judge_same)),s=55,color=COLORS[3],label='Custody marker')
ax.set_yticks(range(len(judge_same)),[f'{j} (n={n})' for j,n in zip(judge_same.index,judge_same.n)])
ax.set(title=f'Same court and charge: {common_code}',xlabel='Marker share of disposed charge rows (%)');ax.legend()
finish(fig,'26_judge_same_charge',f'Court: {common_court}. Judges need ≥30 charge rows. No prior record, evidence strength, or random assignment controls.')
note(f'{nonjudge.sum():,} criminal charge rows have a missing or explicitly nonjudge name (such as Clerk--No Judge) and are excluded from judge comparisons, with an audit table retained. '
     f'The restricted comparison holds court and charge code ({common_code}) constant. Differences remain descriptive; the charts do not identify a judge effect. Source-name variants are not merged without verification.')

## 15. Date discrepancies and time from initiation to disposition

**Why these graphs:** processing-time claims are fragile when the source dates contradict each other. The main distribution excludes missing/negative intervals and retains valid long intervals. Recent dispositions mix old and recent filings. A median among resolved charges is not the expected time for all filed cases, because unresolved cases are absent.

In [ ]:
d['initiation_to_disposition_days']=(d.disposition_date-d.initiated_date).dt.days
d['offense_to_disposition_days']=(d.disposition_date-d.offense_date).dt.days
interval_audit=pd.Series({'disposed_rows':len(d),'negative_initiation_to_disposition':int(d.initiation_to_disposition_days.lt(0).sum()),
 'missing_initiation_to_disposition':int(d.initiation_to_disposition_days.isna().sum()),
 'negative_offense_to_disposition':int(d.offense_to_disposition_days.lt(0).sum())},name='rows')
table('disposition_interval_audit',interval_audit)
valid=d.loc[d.initiation_to_disposition_days.ge(0)]
timing=valid.groupby('disposition_year').initiation_to_disposition_days.agg(n='size',median='median',mean='mean',
 p25=lambda s:s.quantile(.25),p75=lambda s:s.quantile(.75))
table('processing_time_by_disposition_year',timing)
fig,axs=plt.subplots(1,2,figsize=(13,5))
axs[0].plot(timing.index,timing['median'],marker='o');axs[0].fill_between(timing.index,timing.p25,timing.p75,alpha=.2)
axs[0].set(title='Time among resolved charges',xlabel='Disposition year',ylabel='Days (median and IQR)')
axs[1].hist(np.log1p(valid.initiation_to_disposition_days),bins=50,color=COLORS[1])
axs[1].set(title='Long tail of recorded processing times',xlabel='log(1 + days)',ylabel='Charge rows')
finish(fig,'27_processing_times','Nonnegative usable initiation-to-disposition intervals only; invalid intervals excluded and counted separately. No unresolved-case duration estimate.')
negative=d.loc[d.initiation_to_disposition_days.lt(0)]
table('negative_intervals_by_court',negative.groupby('court').size().sort_values(ascending=False).to_frame('negative_rows'))
note(f'{len(negative):,} of {len(d):,} disposed rows ({len(negative)/len(d):.1%}) have disposition before initiation. The valid nonnegative charge-weighted median is {valid.initiation_to_disposition_days.median():,.0f} days. The chronology issue prevents treating the unfiltered mean as reliable.')

## 16. Bail: events, cases, and what the status actually says

**Why these graphs:** show the full recorded status landscape before reducing it to cash/no-cash. A status describes an observation at one time. Bail set is not necessarily posted. Held statuses do not establish inability to pay, and the available history does not measure days detained. Primary first-event views use the earliest observed event **within 2017–2024**, not a guaranteed arraignment.

In [ ]:
b['bail_status']=b.bail_status.fillna('Missing').str.replace(r'\s+',' ',regex=True)
b['bail_status']=b.bail_status.replace({'Out On Cash Bail':'Out on Cash Bail'})
cash_statuses=['Cash Only','Cash/Surety','In Custody - Bail Set','Out on Cash Bail']
def bail_family(status):
    if status=='Released on Own Recognizance':return 'Own recognizance'
    if status in cash_statuses:return 'Cash-related status'
    if status=='Conditions Only' or 'released w/conditions' in status:return 'Conditions / released'
    if status=='Held Without Right To Bail':return 'Held without right to bail'
    if status in ['Held Pending DH','Held as a Danger']:return 'Dangerousness / pending hearing'
    if status=='Bail Revoked':return 'Bail revoked'
    return 'Other recorded status'
b['status_family']=b.bail_status.map(bail_family)
b['cash_status']=b.bail_status.isin(cash_statuses)
b['positive_set']=b.set_amount.gt(0)
b['event_id']=b.source_sheet+' / '+b.source_row.astype(str)
event_status=b.bail_status.value_counts().to_frame('events')
event_status['percent']=event_status.events/len(b)*100
table('bail_raw_status_distribution',event_status,rows=40)
fig,ax=barh(event_status.events.head(16),'What the bail records actually describe','Bail/status events')
finish(fig,'28_bail_statuses','Events dated 2017–2024; one case can appear repeatedly. Cash-related status and recorded positive amount are separate measures.')
first_bail=b.sort_values(['bail_date','bail_status','source_row']).drop_duplicates('case_id').copy()
tie_cases=b.merge(b.groupby('case_id').bail_date.min().rename('first_date'),on='case_id')
tie_count=int((tie_cases.loc[tie_cases.bail_date.eq(tie_cases.first_date)].groupby('case_id').size()>1).sum())
status_year=pd.crosstab(first_bail.bail_date.dt.year,first_bail.status_family)
table('first_observed_bail_status_by_year',status_year)
fig,ax=stacked(status_year,'First observed bail/status event per case: annual mix',percent=True)
ax.set_ylabel('Case IDs within first-observed-event year (%)')
finish(fig,'29_first_bail_status_year','First event in the study window; ties resolved deterministically by status text/source row. This is not guaranteed to be the initial hearing.')
events_per_case=b.groupby('case_id').size()
table('bail_events_per_case_summary',events_per_case.describe().to_frame('events'))
note(f'{len(b):,} bail events represent {b.case_id.nunique():,} case IDs. {tie_count:,} cases have multiple events on their earliest observed date, so first-event ordering is uncertain for those cases. No actual payment or custody-duration conclusion is made.')

## 17. Which crime types have the most cash-related bail?

**Why these graphs:** distinguish total cash-related workload from the within-crime-type rate. The crime type comes directly from the bail file; it is not a conviction category and may summarize a multi-charge case. The primary comparison uses first observed events per case, avoiding repeated-event weighting. Types require at least 100 cases in the composition display.

In [ ]:
first_bail['crime_type']=first_bail.crime_type.fillna('Missing')
crime_status=pd.crosstab(first_bail.crime_type,first_bail.status_family)
crime_stats=first_bail.groupby('crime_type').agg(cases=('case_id','size'),cash_status_cases=('cash_status','sum'),
 positive_set_cases=('positive_set','sum'),set_recorded=('set_amount','count'))
crime_stats['cash_status_pct']=crime_stats.cash_status_cases/crime_stats.cases*100
crime_stats['positive_set_pct']=crime_stats.positive_set_cases/crime_stats.cases*100
crime_stats['set_missing_pct']=(1-crime_stats.set_recorded/crime_stats.cases)*100
table('crime_type_first_bail_summary',crime_stats.sort_values('cases',ascending=False),rows=35)
types=crime_stats.loc[crime_stats.cases>=100].nlargest(12,'cases').index
fig,ax=stacked(crime_status.reindex(types), 'Bail status composition by recorded crime type',percent=True)
ax.set_ylabel('First observed case events (%)')
finish(fig,'30_crime_bail_status_mix','One first observed event per case ID; types require ≥100 cases. Differences are unadjusted for severity, prior record, and hearing context.')
rank=crime_stats.loc[crime_stats.cases>=100].sort_values('cash_status_cases',ascending=False).head(12)
fig,axs=plt.subplots(1,2,figsize=(13,6))
rank.cash_status_cases.sort_values().plot.barh(ax=axs[0],color=COLORS[0]);axs[0].set(title='Largest cash-related caseload',xlabel='Case IDs')
rank.cash_status_pct.sort_values().plot.barh(ax=axs[1],color=COLORS[2]);axs[1].set(title='Within-type cash-related share',xlabel='Percent of first observed case events')
finish(fig,'31_crime_cash_volume_vs_share','Same eligible types. A type with the largest number need not have the highest within-type proportion. Status is not payment.')
cash_sub=first_bail.loc[first_bail.cash_status]
crime_money=cash_sub.groupby('crime_type').agg(cash_status_cases=('case_id','size'),set_recorded=('set_amount','count'),
 median_recorded_set=('set_amount','median'))
crime_money['missing_set_pct']=(1-crime_money.set_recorded/crime_money.cash_status_cases)*100
crime_money['reportable_median']=crime_money.median_recorded_set.where(
 (crime_money.missing_set_pct<=20)&(crime_money.set_recorded>=MIN_CELL))
table('crime_type_cash_status_amount_coverage',crime_money.sort_values('cash_status_cases',ascending=False),rows=35)
reportable=crime_money.reportable_median.dropna().sort_values(ascending=False).head(12)
fig,ax=barh(reportable,'Cash-related first events: reportable median set amounts','Median recorded dollars')
finish(fig,'32_crime_median_cash_amount','Median shown only when ≥30 amounts and ≤20% missing among first-event cash-status cases of that type; zeros retained. No collected-money measure.')
note(f'The largest first-event cash-related caseload is {crime_stats.cash_status_cases.idxmax()}. The median-amount chart applies the team coverage threshold within cash-related statuses, rather than across all recognizance/held events.')

## 18. Requested versus set: how often are they different?

**Why these graphs:** directly compare the prosecutor-request field with the recorded set amount only where both exist. The first chart shows all paired events; a one-paired-event-per-case table assesses repeated-event sensitivity. The first paired event may occur later than the first status event. Missing requests are not zero requests, and these selected observations do not describe every hearing.

In [ ]:
paired=b.loc[b.requested.notna()&b.set_amount.notna()&b.requested.ge(0)&b.set_amount.ge(0)].copy()
paired['comparison']=np.select([paired.set_amount.lt(paired.requested),paired.set_amount.eq(paired.requested)],
                              ['Set lower','Set equal'],default='Set higher')
paired['difference']=paired.set_amount-paired.requested
comparison=paired.comparison.value_counts().reindex(['Set lower','Set equal','Set higher'],fill_value=0).to_frame('events')
comparison['percent']=comparison.events/len(paired)*100
table('requested_vs_set_paired_events',comparison)
pair_case=paired.sort_values(['bail_date','source_row']).drop_duplicates('case_id')
table('requested_vs_set_first_paired_event',pair_case.comparison.value_counts().to_frame('cases'))
fig,axs=plt.subplots(1,2,figsize=(13,5))
axs[0].scatter(paired.requested,paired.set_amount,alpha=.2,s=13,color=COLORS[0])
limit=max(paired.requested.max(),paired.set_amount.max())
axs[0].plot([0,limit],[0,limit],ls='--',color=COLORS[3])
axs[0].set_xscale('symlog',linthresh=100);axs[0].set_yscale('symlog',linthresh=100)
axs[0].set(title='Recorded request and set amount',xlabel='Requested dollars (symlog)',ylabel='Set dollars (symlog)')
axs[1].bar(comparison.index,comparison.percent,color=[COLORS[1],COLORS[0],COLORS[3]])
axs[1].set(title='Direction of the recorded difference',ylabel='Percent of paired events')
finish(fig,'33_requested_vs_set','Only events with both nonnegative amounts. Diagonal = equal. Symlog retains zeros. This does not isolate a judge or establish prosecutor/judge disagreement reasons.')
coverage_pairs=pd.Series({'all_events':len(b),'request_recorded':int(b.requested.notna().sum()),
 'set_recorded':int(b.set_amount.notna().sum()),'paired_events':len(paired),'distinct_paired_cases':pair_case.case_id.nunique(),
 'request_zero':int(b.requested.eq(0).sum()),'set_zero':int(b.set_amount.eq(0).sum()),
 'negative_request':int(b.requested.lt(0).sum()),'negative_set':int(b.set_amount.lt(0).sum())},name='count')
table('bail_amount_and_pair_coverage',coverage_pairs)
note(f'Both amounts are recorded on {len(paired):,} events ({len(paired)/len(b):.1%} of all bail events). '
     f'Set amounts are lower on {comparison.loc["Set lower","percent"]:.1f}% of those paired events. This is a selected-subset description; it is insufficient evidence for an all-hearings rate when coverage is poor.')

## 19. Monetary conditions: distribution and concentration, not revenue

**Why these graphs:** a small number of large orders can dominate monetary totals. Avoid adding every repeated event as if each were a new payment. Select the earliest observed cash-related event with a positive set amount per case. The sum is a descriptive index of recorded monetary conditions across distinct cases, not cash received, outstanding exposure at one time, collections, profit, or forfeitures. Missing amounts make it incomplete.

In [ ]:
positive_cash=b.loc[b.cash_status&b.set_amount.gt(0)].copy()
first_positive=positive_cash.sort_values(['bail_date','source_row']).drop_duplicates('case_id')
money=first_positive.set_amount
table('first_positive_cash_set_distribution',money.describe(percentiles=[.25,.5,.75,.9,.95,.99]).to_frame('dollars'))
money_year=first_positive.groupby(first_positive.bail_date.dt.year).agg(cases=('case_id','size'),
 median_set=('set_amount','median'),sum_recorded_conditions=('set_amount','sum'))
table('first_positive_cash_set_by_year',money_year)
fig,axs=plt.subplots(1,2,figsize=(13,5))
axs[0].hist(np.log10(money),bins=35,color=COLORS[0]);axs[0].set(title='Positive recorded cash-related amounts',xlabel='log10(dollars)',ylabel='Distinct case IDs')
vals=np.sort(money.values);x=np.arange(len(vals)+1)/len(vals);y=np.r_[0,np.cumsum(vals)/vals.sum()]
axs[1].plot(x*100,y*100,color=COLORS[2],lw=2);axs[1].plot([0,100],[0,100],ls='--',color='#999')
axs[1].set(title='Concentration of recorded monetary conditions',xlabel='Cases, ascending amount (%)',ylabel='Cumulative sum of selected amounts (%)')
finish(fig,'34_bail_money_distribution','One earliest observed positive cash-related set amount per case. No payment data; no inference about actual funds received.')
top_decile=max(1,int(np.ceil(len(money)*.1)))
note(f'For {len(money):,} cases with a selected positive cash-related set amount, the median is ${money.median():,.0f}. '
     f'The largest 10% of selected amounts account for {money.nlargest(top_decile).sum()/money.sum():.1%} of their sum. '
     f'The selected-condition sum is ${money.sum():,.0f}; **actual money paid or received is not measurable from these data**.')

## 20. Bail patterns by race and recorded sex

**Why these graphs:** compare the recorded first-event status mix, not the number of events per defendant. The bail file has no person ID, so the unit is case ID. Missing/Unknown remain explicit. Cash-related status and positive recorded amount are not equivalent; neither proves inability to pay. All differences are unadjusted.

In [ ]:
for group in ['race','sex']:
    counts=pd.crosstab(first_bail[group],first_bail.status_family)
    table(f'first_bail_status_by_{group}',counts)
    eligible=counts.sum(axis=1).loc[lambda s:s>=MIN_CELL].index
    fig,ax=stacked(counts.loc[eligible],f'First recorded bail status within {group}',percent=True)
    ax.set_ylabel('First observed case events (%)')
    finish(fig,f'35_bail_{group}_status','One observed event per case ID in 2017–2024. n≥30 groups; Unknown/Missing retained. No prior-record or ability-to-pay adjustment.')
    rows=[]
    for label, sub in first_bail.groupby(group):
        cash=sub.loc[sub.cash_status]
        coverage=100*(1-cash.set_amount.notna().mean()) if len(cash) else np.nan
        rows.append({'group':label,'cases':len(sub),'cash_status_pct':sub.cash_status.mean()*100 if len(sub)>=MIN_CELL else np.nan,
          'cash_status_cases':len(cash),'cash_set_missing_pct':coverage,
          'reportable_median_set':cash.set_amount.median() if len(cash)>=MIN_CELL and coverage<=20 else np.nan})
    table(f'first_bail_{group}_money_coverage',pd.DataFrame(rows),rows=30)
note('These charts describe recorded case experiences. They cannot distinguish differences in case severity, pretrial history, finances, or systemic decision-making. A detention disparity cannot be calculated from cash-bail status alone.')

## 20A. Cash-only, cash/surety, and recorded status changes

**Why these graphs:** grouping every monetary status together can hide meaningful differences. `Cash Only` and `Cash/Surety` are retained exactly as recorded. A surety is a security arrangement; the extract does not identify the actual instrument, deposit, or who supplied it. No assumption is made that a person paid 10% or the full amount. `In Custody - Bail Set` differs from `Out on Cash Bail`, but neither supplies a detention duration.

The transition view compares the earliest and latest observed status **within the window**, only for cases with observations on different dates. It is a summary of endpoints, not every intermediate transition or a final case result.

In [ ]:
raw_crime_status=pd.crosstab(first_bail.crime_type,first_bail.bail_status)
table('crime_type_vs_raw_bail_status_counts',raw_crime_status,rows=35)
raw_crime_shares=raw_crime_status.div(raw_crime_status.sum(axis=1),axis=0)*100
table('crime_type_vs_raw_bail_status_shares_pct',raw_crime_shares,rows=35)
eligible_types=raw_crime_status.sum(axis=1).loc[lambda s:s>=100].index
raw_cash=raw_crime_status.reindex(eligible_types).reindex(columns=['Cash Only','Cash/Surety'],fill_value=0)
raw_cash=raw_cash.loc[raw_cash.sum(axis=1).nlargest(12).index]
fig,axs=plt.subplots(1,2,figsize=(14,6))
for ax,col,color in zip(axs,['Cash Only','Cash/Surety'],[COLORS[0],COLORS[2]]):
    raw_cash[col].sort_values().plot.barh(ax=ax,color=color)
    ax.set(title=f'Recorded {col} status by crime type',xlabel='First observed case events',ylabel='')
finish(fig,'39_cash_only_vs_surety','One first observed event per case. Crime types need ≥100 total cases; these are recorded statuses, not payment totals.')
first_date=b.groupby('case_id').bail_date.min()
last_date=b.groupby('case_id').bail_date.max()
longitudinal_ids=first_date.index[last_date.gt(first_date)]
last_bail=b.sort_values(['bail_date','bail_status','source_row']).drop_duplicates('case_id',keep='last')
endpoints=first_bail.loc[first_bail.case_id.isin(longitudinal_ids),['case_id','status_family']].merge(
 last_bail[['case_id','status_family']],on='case_id',suffixes=('_first','_last'),validate='one_to_one')
transition_counts=pd.crosstab(endpoints.status_family_first,endpoints.status_family_last)
table('first_to_last_observed_status_counts',transition_counts)
transition_shares=transition_counts.div(transition_counts.sum(axis=1),axis=0)*100
transition_display=transition_shares.where(transition_counts>=MIN_CELL)
fig,ax=heatmap(transition_display,'First to last observed status among repeatedly observed cases',percent=True,annotate=True)
ax.set(xlabel='Last observed status family',ylabel='First observed status family')
finish(fig,'40_bail_recorded_status_endpoints','Only cases with observations on different dates. Row percentages; cells under 30 cases suppressed. Endpoint ordering within a tied day is uncertain.')
crime_pairs=paired.groupby('crime_type').agg(paired_events=('case_id','size'),cases=('case_id','nunique'),
 median_request=('requested','median'),median_set=('set_amount','median'),
 median_set_minus_request=('difference','median'))
crime_pairs['set_lower_pct']=paired.groupby('crime_type').comparison.apply(lambda s:s.eq('Set lower').mean()*100)
crime_pairs.loc[crime_pairs.paired_events<MIN_CELL,['median_request','median_set','median_set_minus_request','set_lower_pct']]=np.nan
table('crime_type_requested_vs_set_paired_subset',crime_pairs.sort_values('paired_events',ascending=False),rows=35)
note(f'{len(endpoints):,} case IDs have bail/status records on different dates in the window. Endpoint changes describe the available records, not the duration or cause of detention. Cash-only and cash/surety counts are available separately for every recorded crime type.')

## 21. Link filing and disposition records without multiplying rows

**Why this analysis:** matched records can connect allegations to recorded outcomes, but an unrestricted many-to-many merge would inflate counts. Match on case ID + defendant ID + charge count, validate one-to-one keys after removing exact repeated exports for linkage only, and then require the charge code to agree and disposition to occur on or after initiation. Ambiguous keys, mismatched codes, and negative chronology are reported separately. Unmatched records remain in the cohort denominator.

The earlier overview counted unmatched **case IDs**, not missing people. The actual counts depend on date window and matching level; they are recomputed here. An absent disposition can mean pending, later resolution, extraction coverage, changed identifiers, or coding differences.

In [ ]:
charge_key=['case_id','person_id','count_no']
def linkage_unique(frame):
    clean=frame.loc[~frame.exact_duplicate].copy()
    ambiguous=clean.duplicated(charge_key,keep=False)
    return clean.loc[~ambiguous].copy(),clean.loc[ambiguous].copy()
left,left_ambiguous=linkage_unique(f)
right,right_ambiguous=linkage_unique(disposed.loc[disposed.disposition_date.le(FOLLOWUP_END)])
keep=charge_key+['charge_code','disposition_date','outcome_detail','disp_type','court','judge']
linked=left.merge(right[keep],on=charge_key,how='left',suffixes=('_filed','_disposed'),validate='one_to_one',indicator=True)
linked['same_code']=linked.charge_code_filed.eq(linked.charge_code_disposed).fillna(False)
linked['chronology_valid']=linked.disposition_date.ge(linked.initiated_date)
linked['compatible_2024']=linked.same_code&linked.chronology_valid&linked.disposition_date.le(END)
linked['compatible_2025']=linked.same_code&linked.chronology_valid&linked.disposition_date.le(FOLLOWUP_END)
link_audit=pd.Series({'filed_rows_primary':len(f),'filed_rows_after_exact_repeat_removal':int((~f.exact_duplicate).sum()),
 'ambiguous_filed_key_rows_quarantined':len(left_ambiguous),'ambiguous_disposition_key_rows_quarantined':len(right_ambiguous),
 'eligible_filed_charge_keys':len(left),'matched_key_any_followup':int(linked['_merge'].eq('both').sum()),
 'matched_key_different_charge_code':int((linked['_merge'].eq('both')&~linked.same_code).sum()),
 'matched_same_code_disposition_before_initiation':int((linked.same_code&linked.disposition_date.lt(linked.initiated_date)).sum()),
 'same_code_disposition_by_2024':int(linked.compatible_2024.sum()),
 'same_code_disposition_by_2025':int(linked.compatible_2025.sum()),
 'eligible_without_compatible_disposition_by_2024':int((~linked.compatible_2024).sum())},name='rows_or_keys')
table('charge_linkage_audit',link_audit)
fs=set(f.case_id.dropna());ds=set(d.case_id.dropna())
table('case_id_overlap_study_windows',pd.Series({'filed_case_ids':len(fs),'disposed_case_ids':len(ds),
 'intersection':len(fs&ds),'disposed_only':len(ds-fs),'filed_only':len(fs-ds)},name='case_ids'))
year_link=linked.groupby(linked.initiated_date.dt.year).agg(eligible_charge_keys=('case_id','size'),
 observed_by_2024=('compatible_2024','sum'),observed_by_2025=('compatible_2025','sum'))
year_link['share_by_2024_pct']=year_link.observed_by_2024/year_link.eligible_charge_keys*100
year_link['share_by_2025_pct']=year_link.observed_by_2025/year_link.eligible_charge_keys*100
table('linked_filing_cohort_followup_by_year',year_link)
fig,ax=plt.subplots(figsize=(11,5))
for col,label in [('share_by_2024_pct','Disposition recorded by 2024-12-31'),('share_by_2025_pct','Sensitivity: recorded by 2025-12-31')]:
    ax.plot(year_link.index,year_link[col],marker='o',label=label)
ax.set(title='Observed compatible dispositions by filing cohort',xlabel='Initiation year',ylabel='Percent of eligible filed charge keys',ylim=(0,100));ax.legend(fontsize=9)
finish(fig,'36_linked_cohort_followup','Unambiguous same-code keys with disposition on/after initiation. Missing or invalid match is not automatically pending. Recent cohorts have shorter follow-up.')
observed=linked.loc[linked.compatible_2024]
table('linked_charge_outcomes_by_2024',observed.outcome_detail.value_counts().to_frame('matched_charge_keys'),rows=30)
note(f'{linked.compatible_2024.sum():,} of {len(linked):,} unambiguous filed charge keys have a same-code, chronologically valid disposition recorded by the 2024 cutoff. '
     f'{len(left_ambiguous):,} filed rows were quarantined for ambiguous keys. Matching does not justify dropping unmatched cases from all population-level conclusions.')

## 22. Link bail to filing cases and recorded outcomes

**Why this analysis:** connect a case-level bail observation to filing severity and charge burden without joining each bail event to every charge. First aggregate filings to one row per case ID. Exclude case IDs associated with multiple defendant IDs because bail has no person ID. For the outcome association, use only disposition rows on or after the selected bail date, then aggregate to one row per case ID with transparent mixed-outcome and custody markers. This is a selected matched cohort, not an estimate of the effect of bail on conviction.

In [ ]:
case_people=filed.groupby('case_id').person_id.nunique()
single_person_ids=case_people.loc[case_people==1].index
rank_map={'F':4,'M+':3,'M-':2,'CM':1,'Unknown':0}
ordered_f=f.assign(class_rank=f.offense_class.map(rank_map)).sort_values(
 ['case_id','class_rank','charge_code','count_no'],ascending=[True,False,True,True])
representative=ordered_f.drop_duplicates('case_id').set_index('case_id')
filing_case=f.groupby('case_id').agg(charge_rows=('case_id','size'),person_count=('person_id','nunique'),
 initiation=('initiated_date','min'),any_criminal_charge=('criminal_charge','max'))
filing_case['representative_class']=representative.offense_class
filing_case['representative_charge_code']=representative.charge_code
filing_case=filing_case.loc[filing_case.index.isin(single_person_ids)]
temporal=d.merge(first_bail[['case_id','bail_date']],on='case_id',how='inner',validate='many_to_one')
before_bail=temporal.disposition_date.lt(temporal.bail_date)
table('bail_outcome_temporal_audit',pd.Series({'disposition_rows_sharing_a_bail_case':len(temporal),
 'disposition_rows_before_first_observed_bail':int(before_bail.sum()),
 'rows_on_or_after_first_bail':int((~before_bail).sum())},name='charge_rows'))
outcome_case=temporal.loc[~before_bail].groupby('case_id').agg(any_custody_marker=('custody_marker','max'),any_CWOF_marker=('cwof_marker','max'),
 outcome_families=('outcome_family','nunique'),disposed_charge_rows=('case_id','size'))
joined=first_bail.merge(filing_case,on='case_id',how='inner',validate='one_to_one').merge(
 outcome_case,on='case_id',how='left',validate='one_to_one')
joined['observed_disposition']=joined.disposed_charge_rows.notna()
joined['charge_burden']=pd.cut(joined.charge_rows,bins=[0,1,2,4,np.inf],labels=['1 charge','2 charges','3–4 charges','5+ charges'])
table('bail_case_linkage_coverage',pd.Series({'first_observed_bail_cases':len(first_bail),
 'filing_case_ids_single_person':len(filing_case),'matched_bail_filing_cases':len(joined),
 'matched_with_disposition_in_window':int(joined.observed_disposition.sum()),
 'case_ids_with_multiple_people_in_full_filings':int((case_people>1).sum())},name='case_ids'))
burden=pd.crosstab(joined.charge_burden,joined.status_family)
table('bail_status_by_filing_charge_burden',burden)
fig,ax=stacked(burden,'First bail status and recorded charge burden',percent=True)
ax.set_ylabel('Matched case IDs within charge-burden group (%)')
finish(fig,'37_bail_charge_burden','Case-level one-to-one linkage; multi-defendant case IDs excluded. Charge burden is observed across the filing extract, not necessarily all known at the bail event.')
matched_outcomes=joined.loc[joined.observed_disposition].copy()
association=matched_outcomes.groupby('status_family').agg(cases=('case_id','size'),
 custody_cases=('any_custody_marker','sum'),CWOF_cases=('any_CWOF_marker','sum'))
association['custody_marker_pct']=association.custody_cases/association.cases*100
association['CWOF_marker_pct']=association.CWOF_cases/association.cases*100
association.loc[association.cases<MIN_CELL,['custody_marker_pct','CWOF_marker_pct']]=np.nan
table('matched_bail_outcome_associations',association)
match_coverage=joined.groupby('status_family').agg(cases=('case_id','size'),
 dispositions_observed=('observed_disposition','sum'))
match_coverage['disposition_not_observed_pct']=(1-match_coverage.dispositions_observed/match_coverage.cases)*100
table('bail_outcome_linkage_missingness',match_coverage)
# Plot outcome-dependent rates only where outcome linkage meets the team's 20% rule.
eligible=match_coverage.index[(match_coverage.disposition_not_observed_pct<=20)&(match_coverage.cases>=MIN_CELL)]
report=association.reindex(eligible)[['custody_marker_pct','CWOF_marker_pct']].dropna(how='all')
if len(report):
    fig,ax=plt.subplots(figsize=(11,5));report.plot.bar(ax=ax,color=[COLORS[3],COLORS[1]])
    ax.set(title='Recorded outcomes among linked cases, by bail status',ylabel='Percent of cases with an observed disposition',xlabel='')
    ax.tick_params(axis='x',rotation=30)
    finish(fig,'38_bail_outcome_association','Dispositions on/after first observed bail; ≥30 cases and ≤20% missing linkage per status. Markers can overlap. No causal bail effect.')
else:
    display(Markdown('**Insufficient data:** no bail-status group meets the outcome-linkage coverage rule for a comparative outcome-rate chart.'))
note('The linked view avoids multiplying bail events by charges and keeps coverage visible. Custody and CWOF case markers can coexist; they are not mutually exclusive final case judgments. The comparison cannot establish whether bail causes later outcomes.')

## 23. Robustness, limitations, and questions for the client

The following checks reconcile denominators and prevent silent population changes. They verify transformations rather than asserting a preferred substantive finding.

In [ ]:
assert annual.charge_rows.sum()==len(f)
assert classes.to_numpy().sum()==len(f)
assert raw_outcomes.charge_rows.sum()==len(d)
assert plea_table.charge_rows.sum()==len(pleas)
assert len(first_bail)==b.case_id.nunique()
assert len(linked)==len(left)
assert linked.compatible_2024.sum()<=linked.compatible_2025.sum()
assert len(joined)==joined.case_id.nunique()
assert all(hashlib.sha256(path.read_bytes()).hexdigest()==source_hashes[name] for name,path in paths.items())
sensitivity=[]
for name,frame in [('Filed',f),('Disposed',d),('Bail',b)]:
    sensitivity.append({'population':name,'primary_rows':len(frame),
      'without_exact_repeated_exports':int((~frame.exact_duplicate).sum()),
      'difference_pct':frame.exact_duplicate.mean()*100})
table('exact_repeat_sensitivity',pd.DataFrame(sensitivity))
table('raw_demographic_code_counts_filed',pd.concat({'race':f.race_code.fillna('Missing').value_counts(),
 'sex':f.sex_code.fillna('Missing').value_counts()},axis=1).fillna(0))
table('class_code_counts_filed',f.offense_class.value_counts().to_frame('charge_rows'))
print('PASS: population totals, plea totals, unique first-bail cases, one-to-one links, follow-up monotonicity, and unchanged source SHA-256 hashes.')

### Interpretation limits that apply to the whole notebook

- **Charge-level is the main unit requested here.** Charge-row percentages differ from case/defendant percentages. Case-level linked views explicitly aggregate first. A case can carry mixed outcomes.
- **Source population:** these are supplied DA extracts, not all crimes, arrests, referrals, or residents. Declination rates, offense prevalence, and per-capita prosecution rates are not identified here.
- **Time:** filing-year, disposition-year, and bail-event-year populations differ. Later extraction information is not a historical snapshot of case status. Recent filing cohorts have shorter follow-up. A disposed extract alone cannot measure unresolved-case delays.
- **Dates:** no century is guessed. Invalid/missing dates and negative durations are counted. Plausible long intervals remain in primary summaries; sensitivity restrictions are separately labeled. A disposition before initiation can reflect data entry, import, changed case identifiers, or misunderstood field semantics; its cause cannot be proven from the workbook.
- **Repeated rows and keys:** exact repeated exports remain in primary workload counts; a removal sensitivity is provided. Charge linkage removes exact repeats only for matching, quarantines nonunique keys, and requires charge-code agreement. Changed charge codes are not assumed to represent charging upgrades/downgrades.
- **Classification:** source offense class is not independently validated against each historical statute. Proposed outcome families retain unresolved/conflicting records and require client review. No all-pleas-equal-convictions assumption is made. A recorded custody marker does not prove time served, facility, or sentence duration.
- **Missingness:** blank, explicit Unknown, and unverified codes are distinguished. The team 20% rule is applied within relevant eligible populations for dependent rate claims. Conditional available-record summaries are labeled as selected subsets; they are not full-population estimates.
- **People and equity:** charges are allegations. Internal IDs are county-specific; no cross-county identity inference. Recorded sex and race categories are administrative measures. Demographic comparisons lack resident populations, prior record, finances, counsel, evidence strength, and complete case facts. Small demographic groups and controlled-comparison cells are suppressed as stated on each chart. Descriptive differences do not establish unlawful discrimination.
- **Judges:** recorded names need verification. Case assignment, court, charge mix, negotiated outcomes, prosecutorial decisions, and prior proceedings prevent causal attribution. Same-court/same-charge restriction is a sensitivity, not a complete adjustment or performance ranking.
- **Bail:** no deposits, payment receipts, forfeitures, refunds, household income, or reliable time-in-custody history are present. Requested/set amounts cannot answer “how much money the county received.” First observed event may be late in the history; same-day ordering is uncertain. Crime type may summarize a multi-charge case. No 10%-bond assumption is made.
- **Linkage:** missing case IDs are not missing people. Never interpret an unmatched record as definitely pending. Matched-only outcomes can be selected; coverage by bail status is reported and low-coverage rate plots are withheld.

### Questions that would materially improve the analysis

1. What do initiation date, disposition type, and disposition description represent in Berkshire’s export, and why can their chronology/labels conflict?
2. Can the DA supply a Berkshire race/sex codebook, especially `M`, `BR`, `EI`, `I`, `MNA`, `C`, and sex `N`?
3. Which repeated charge-count keys represent corrections, multiple proceedings, or true duplicates? Can a stable charge-row ID be supplied?
4. Does `Sentence Translation` describe the initial order, current sentence, or last recorded sentence? How should contradictory CWOF/custody markers be resolved?
5. What is the precise meaning and event timing of cash/surety, requested bail, and set bail? Can actual payment and custody histories be provided separately?
6. What is the extraction date, and are the separate filed/disposed extracts complete for their respective date filters?

### How to extend responsibly

After reviewing the proposed crosswalk and flags with the client, add validated statutory seriousness and historical crime-code lookups, population denominators for clearly specified rates, and a documented case outcome hierarchy. A later analysis can estimate associations with more controls and defendant-level uncertainty. This notebook deliberately makes the observable evidence and its limits inspectable first.

In [ ]:
manifest={'study_start':str(START.date()),'study_end':str(END.date()),'followup_sensitivity_end':str(FOLLOWUP_END.date()),
 'sources':{k:{'filename':v.name,'sha256':source_hashes[k]} for k,v in paths.items()},
 'versions':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,
             'matplotlib':plt.matplotlib.__version__,'openpyxl':openpyxl.__version__},
 'figures':FIGURES,'tables':list(TABLES),'source_files_unchanged':True}
(OUT/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
(OUT/'key_findings.md').write_text('# Berkshire EDA: computed findings\n\n'+'\n\n'.join('- '+s for s in FINDINGS),encoding='utf-8')
print(f'Completed: {len(FIGURES)} figures and {len(TABLES)} aggregate tables. Original source hashes unchanged.')
print('Outputs:',OUT)